
```
```
```
```
**ESB2025** **/ AI Based Shape Representation /** **Hands-On Session**
```
```
```
```


In [ ]:
!git clone --branch ESB25_workshop_AI4shp https://github.com/mohofar/PedVision.git

In [ ]:
cd PedVision

In [ ]:
# !pip install -r requirements.txt
# opencv-python==4.8.1.78
# numpy==1.26.3
# matplotlib==3.8.3
# ipython==8.22.2
# scikit-image==0.22.0
# pillow==10.2.0
!pip install segmentation-models-pytorch==0.4.0
# tqdm==4.66.2
# scikit-learn==1.4.1.post1

## Pedvision

In [ ]:
!python PedVisionCode/main.py --foldering y

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P PedVisionCode/saved_models
!wget "https://www.dropbox.com/scl/fi/7kzywiswmslrfmlgcp6vo/ROI_model_R11.pth?rlkey=idpqozialittcri90us2hy1vc&st=usq0igej&dl=1" -O PedVisionCode/saved_models/ROI_model_R11.pth
!wget "https://www.dropbox.com/scl/fi/5p9zymm76fnhj95grmq73/CLS_model_R11.pth?rlkey=urzxlrjrsq1du397za8vucpvy&st=n3fktzg7&dl=1" -O PedVisionCode/saved_models/CLS_model_R11.pth

In [ ]:
import matplotlib.pyplot as plt
import cv2
image_name1 = '/content/PedVision/PedVisionCode/test_data/input/2428.jpg'
image1 = cv2.imread(image_name1)
plt.imshow(image1)
plt.show()

In [ ]:
!python PedVisionCode/main.py\
--test_model y\
--img_name 2428

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


img_name = os.path.basename(image_name1)[:-4]
cls_num = 5
mask_path = f'PedVisionCode/test_data/predicted/VFM/org_mask_{img_name}.pkl'
with open(mask_path, 'rb') as f:
    masks = pickle.load(f)
# print(masks[0].keys())
# print(masks[0]['crop_box'])
mask = np.zeros(image1.shape)


prediction = np.load(f'PedVisionCode/test_data/prepared/{img_name}.npy')
# Visualize predicted masks
plt.figure(figsize=(30,30))
# Display each class's mask
for bone in range(cls_num):
    overall_mask = np.zeros(image1.shape[:2])
    for i, pred in enumerate(prediction):
        if pred.item() == bone:
            overall_mask[masks[0]['crop_box'][0]:masks[0]['crop_box'][1], masks[0]['crop_box'][2]:masks[0]['crop_box'][3]] += masks[i]['segmentation']

    plt.subplot(1, cls_num, bone + 1)
    plt.imshow(image1)
    plt.imshow(overall_mask.astype(np.bool_), alpha=0.7)
    plt.title(f'Class {bone}')
    # plt.axis('off')

plt.show()

### Noise Introduction

In [ ]:
from PedVisionCode.utils import noise_add

# show_noise_examples(image1, 'gaussian', [90, 70, 50, 20])
noise_add.show_noise_examples(image1, 'salt_pepper', 60)
# quick_noise_demo(image1, 'motion')
noise_add.compare_all_noise_types(image1, 60)

In [ ]:
noise_add.save_noise_image('/content/PedVision/PedVisionCode/test_data/input/1422.jpg', '/content/PedVision/PedVisionCode/test_data/input/' , 'salt_pepper', 80)

In [ ]:
!python PedVisionCode/main.py --test_model y\
--img_name 1473

In [ ]:
img_path = '/content/PedVision/PedVisionCode/test_data/input/1422_salt_pepper_N_lvl_40.jpg'
noise_add.visualize_masks(img_path)

```
```
Noise + VFM modifcation
```
```

In [ ]:
!python PedVisionCode/main.py --test_model y --round 11 --num_classes 5\
--img_name 1422_salt_pepper_N_lvl_40\
--points_per_side=42\
--stability_score_thresh=0.8\
--CLS_model_name EffiB5
# --points_per_side=32\
# --pred_iou_thresh=0.5\
# --crop_n_layers=1\

In [ ]:
img_path = '/content/PedVision/PedVisionCode/test_data/input/1422_salt_pepper_N_lvl_40.jpg'
noise_add.visualize_masks(img_path)

## Non-VFM based

In [ ]:
# from torch.utils.data import random_split
# import torchvision.transforms.functional as TF
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# import os
# from glob import glob
# import torch
# from torch import nn
from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
import numpy as np
from PIL import Image
import os
import cv2

import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
# import random


# from skimage.morphology import convex_hull_image
# from skimage.transform import resize


import matplotlib.pyplot as plt
# from torch.utils.data import random_split

from tqdm import tqdm
import segmentation_models_pytorch as smp

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        # self.mask_dir = mask_dir
        self.transform = transform
        self.images = []
        for img in os.listdir(image_dir):
            if img.endswith('.png') or img.endswith('.jpg'):
                self.images.append(img)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        # mask_path = os.path.join(self.mask_dir, self.images[idx][:-4] + '.png')
        # Read the image using OpenCV
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Convert the image from NumPy array to a PIL Image
        image = Image.fromarray(image)

        # Load the mask
        # mask = cv2.imread(mask_path)

        # Define the colormap
        # colors = [[0, 0, 128], [0, 128, 0], [128, 0, 0], [0, 128, 128]]

        # Create binary masks for each color channel
        # binary_masks = []
        # for color in colors:
        #     color_mask = cv2.inRange(mask, np.array(color), np.array(color))
        #     binary_masks.append(color_mask)

        # Stack the binary masks to create a 4-channel mask
        # binary_mask = np.stack(binary_masks, axis=-1)

        # Convert the 4-channel mask to a PIL Image
        # binary_mask = Image.fromarray(binary_mask)

        if self.transform is not None:
            image = self.transform(image)

        return image, os.path.basename(self.images[idx])



class CustomTransformTest:
    def __init__(self):
        # Separate resize transforms for image and mask
        self.resize_image = transforms.Resize((1024, 1024))
        # self.resize_mask = transforms.Resize((1024, 1024), interpolation=transforms.InterpolationMode.NEAREST)


        # Convert to grayscale
        self.to_grayscale = transforms.Grayscale()

    def __call__(self, image):
        # Resize image and mask
        image = self.resize_image(image)
        # mask = self.resize_mask(mask)

        # Convert to tensor after all PIL image transformations
        image = TF.to_tensor(image)
        # mask = TF.to_tensor(mask)

        # Convert to grayscale
        image = self.to_grayscale(image)

        return image

In [ ]:
image_dir = "/content/PedVision/PedVisionCode/test_data/input/"
custom_transform = CustomTransformTest()
dataset_te = CustomDataset(image_dir, transform=custom_transform)

print(len(dataset_te))


test_loader = DataLoader(dataset_te, batch_size=1, shuffle=True)
print(len(test_loader))


# for epoch in range(4):
#     plt.figure(figsize=(15, 5))
#     for batch_idx, (data, name) in enumerate(test_loader):
#         print(data.shape)
#         for i in range(1):
#             plt.subplot(2, 4, i + 1)
#             plt.imshow(data[i, 0, :, :], cmap='gray')
#             plt.axis('off')
#             plt.subplot(2, 4, i + 5)
#             # plt.imshow((target[i, 0, :, :]), cmap='gray')
#             # plt.axis('off')
#             plt.title(name[i])
#         plt.show()
#         break

In [ ]:
# model_name = 'deeplab34'
# model_name = 'deeplab101'
# model_name = 'unet_res34'
# model_name = 'unet_res101'
# model_name = 'segformer_mitb0'
# model_name = 'segformer_mitb1'
# model_name = 'segformer_mitb2'
model_name = 'segformer_mitb3'

In [ ]:
# !wget "https://www.dropbox.com/scl/fi/tlowi0405ovrrs0vdz4ic/Segformer_mitb0.pth?rlkey=hagjaz2koa93afgrrvu56gkpn&st=t0lwm7jt&dl=1" -O PedVisionCode/saved_models/Segformer_mitb0.pth
# !wget "https://www.dropbox.com/scl/fi/nv0ra1xko2y8t4b1bt97c/Segformer_mitb1.pth?rlkey=yk7yrdtj878o8usof4qublv1m&st=xcead696&dl=1" -O PedVisionCode/saved_models/Segformer_mitb1.pth
# !wget "https://www.dropbox.com/scl/fi/5mf0fv4htb8vehcsfl6d3/Segformer_mitb2.pth?rlkey=f2df0pv4bo0a5v2zmpfvpanhd&st=wygqqi50&dl=1" -O PedVisionCode/saved_models/Segformer_mitb2.pth
!wget "https://www.dropbox.com/scl/fi/07tggc1sy54oqwlv73evb/Segformer_mitb3.pth?rlkey=rc6nsw2j14fytkxdkjrispc36&st=kkla9ac7&dl=1" -O PedVisionCode/saved_models/Segformer_mitb3.pth
# !wget "https://www.dropbox.com/scl/fi/j375w7rjqx7x0w18c1t09/Unet_res34.pth?rlkey=epomx2uloprl71y0rh4go6h8j&st=j91wsim7&dl=1" -O PedVisionCode/saved_models/Unet_res34.pth
# !wget "https://www.dropbox.com/scl/fi/jc1zod78qtm5pnjrvu77u/Unet_res101.pth?rlkey=5fgq6oqbr3f1q8nnecb77j7lf&st=n3hv56kl&dl=1" -O PedVisionCode/saved_models/Unet_res101.pth
# !wget "https://www.dropbox.com/scl/fi/nw4vtzmlfxtjn0lu1l62m/DeepLabV3Plus_resnet34_best_model.pth?rlkey=qixdcwdps3gxrd253qdbflf3v&st=1wi8hwob&dl=1" -O PedVisionCode/saved_models/DeepLabV3Plus_resnet34_best_model.pth
# !wget "https://www.dropbox.com/scl/fi/omef5qou8fo50nktwyqk6/Deeplapv3p_res101_best_model.pth?rlkey=i4y1uywekm4s6tfciv3wxywiw&st=oq5jyjkz&dl=1" -O PedVisionCode/saved_models/Deeplapv3p_res101_best_model.pth

In [ ]:
def test_model(model, test_loader):
    model.eval()
    outputs = []
    names = []
    with torch.no_grad():
        for inputs, name in tqdm(test_loader):
            output = model(inputs)

            # Convert the output to a NumPy array and append to the list
            outputs.append(output.squeeze().cpu().numpy())
            names.append(name[0])
    return outputs, names


if  model_name == 'deeplab34':
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34", # choose encoder, e.g., resnet34, mobilenet_v2, etc.
        encoder_weights="imagenet", # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1, # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4, # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )

    print(model)

    # load the best model
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/DeepLabV3Plus_resnet34_best_model.pth'))

elif model_name == 'deeplab101':
    model = smp.DeepLabV3Plus(
        encoder_name="resnet101",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1, # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4, # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Deeplapv3p_res101_best_model.pth'))

elif model_name == 'unet_res34':
    model = smp.Unet(
        encoder_name="resnet34",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Unet_res34.pth'))

elif model_name == 'unet_res101':
    model = smp.Unet(
        encoder_name="resnet101",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Unet_res101.pth'))

elif model_name == 'segformer_mitb0':
    model = smp.Segformer(
        encoder_name="mit_b0",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Segformer_mitb0.pth'))

elif model_name == 'segformer_mitb1':
    model = smp.Segformer(
        encoder_name="mit_b1",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Segformer_mitb1.pth'))

elif model_name == 'segformer_mitb2':
    model = smp.Segformer(
        encoder_name="mit_b2",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Segformer_mitb2.pth'))

elif model_name == 'segformer_mitb3':
    model = smp.Segformer(
        encoder_name="mit_b3",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pretrained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for grayscale images, 3 for RGB, etc.)
        classes=4,                      # model output channels (number of classes in your dataset)
        activation='sigmoid'
    )
    print(model)
    model.load_state_dict(torch.load('/content/PedVision/PedVisionCode/saved_models/Segformer_mitb3.pth'))

In [ ]:
# Set paths to your image and mask directories
# image_dir = "unlabelled_samples2/"
custom_transform = CustomTransformTest()
dataset_te = CustomDataset(image_dir, transform=custom_transform)
test_loader = DataLoader(dataset_te, batch_size=1, shuffle=False)
outputs, names = test_model(model, test_loader)

for i in range(len(outputs)):
    print(i,'->', names[i])

In [ ]:
def img_show(case_num):
    print(names[case_num])
    # load the image
    image = cv2.imread(image_dir+names[case_num])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(30,30))
    for i in range(len(outputs[case_num])):
        plt.subplot(1, len(outputs[case_num]), i+1)
        plt.imshow(image)
        plt.imshow(outputs[case_num][i], alpha=0.7)
        plt.title(f'Class {i}')
        plt.axis('off')
    plt.show()

img_show(case_num=6)